# Stage 3 — RAG QA + Evaluation

End-to-end evaluation of the MSADS RAG chatbot.

**Inputs**:
- Stage 2 v2 Chroma DB at `data/vector_store/MSADS_RAG_DB/` (607 chunks, 0 dups)
- `eval_set.json` — 30 manually-authored Q-A pairs across 6 categories

**Pipeline per question**:
1. Retrieve top-k chunks from Chroma using BGE-small embeddings
2. Send to Claude Sonnet 4.6 with grounded-only system prompt — must return JSON `{answer, sources[]}`
3. Parse cited URLs out of the structured response

**Metrics computed**:
- **Retrieval**: R@1 / R@3 / R@5 / R@10 (gold_url ∈ retrieved chunks' URLs)
- **RAGAS** (4): faithfulness, answer_relevancy, context_precision, context_recall
- **URL Citation** (custom, since the requirement is to cite source URLs): citation_recall / citation_precision / citation_f1
- **Per-category breakdown**
- **Edge case handling**: out_of_scope refusal rate, time/image caveat rate

**Output**: `eval_results.csv` (per-question) + `eval_summary.md` (per-category aggregates).


## 0. Environment setup

In [ ]:
# Run once on Colab
!pip install -q langchain langchain-community langchain-chroma langchain-huggingface langchain-anthropic chromadb sentence-transformers ragas datasets pandas


In [ ]:
# Clone repo (skip if already in repo dir)
import os
if not os.path.exists("RAG-based-Interactive-AI-for-MSADS-Website"):
    !git clone -q https://github.com/Yoki-SyZhang/RAG-based-Interactive-AI-for-MSADS-Website.git
%cd RAG-based-Interactive-AI-for-MSADS-Website/
!ls data/vector_store/MSADS_RAG_DB/


In [ ]:
# API key — Anthropic Claude (judge LLM + RAG generator)
# Best: use Colab's userdata for security (Tools → Secrets → ANTHROPIC_API_KEY)
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    # Local fallback — set in env before launch
    if "ANTHROPIC_API_KEY" not in os.environ:
        raise RuntimeError("Set ANTHROPIC_API_KEY in Colab Secrets or env var")
print("✓ Anthropic API key loaded")


## 1. Config

In [ ]:
DB_PATH = "data/vector_store/MSADS_RAG_DB"
EVAL_SET_PATH = "eval_set.json"   # produced by Stage 3 authoring
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
CLAUDE_MODEL = "claude-sonnet-4-20250514"   # generator + judge
RETRIEVAL_K = 5    # top-K chunks per query

# Output paths
RESULTS_CSV = "eval_results.csv"
SUMMARY_MD = "eval_summary.md"


## 2. Load Chroma DB + Eval Set

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import json

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
vector_db = Chroma(
    persist_directory=DB_PATH,
    embedding_function=embeddings,
    collection_name="langchain",
)
print(f"✓ Loaded Chroma DB: {vector_db._collection.count()} chunks")

with open(EVAL_SET_PATH) as f:
    eval_set = json.load(f)
print(f"✓ Loaded eval set: {len(eval_set)} questions")

# Show category distribution
from collections import Counter
print("\nCategory distribution:")
for cat, n in Counter(q["category"] for q in eval_set).items():
    print(f"  {cat}: {n}")
print("\nSub-type distribution:")
for st, n in Counter(q["sub_type"] for q in eval_set).items():
    print(f"  {st}: {n}")


## 3. RAG chain — grounded only, JSON output schema

The generator MUST return strict JSON `{answer, sources}` so we can parse cited URLs.

In [ ]:
SYSTEM_PROMPT = """You are an assistant for the University of Chicago MS in Applied Data Science (MS-ADS) program. Answer user questions using ONLY the provided context excerpts from the program's official website.

CRITICAL RULES:
1. Output STRICT JSON in this exact schema:
   {
     "answer": "<2-4 sentence direct answer in plain English; no markdown headers>",
     "sources": [
       {"breadcrumb": "<page breadcrumb>", "url": "<canonical URL>"}
     ]
   }
2. Cite EVERY URL whose context excerpt you used. Do NOT cite URLs you didn't use.
3. If the context does NOT contain the answer, output:
   {"answer": "I don't have that information in the program website. Please contact applieddatascience-admissions@uchicago.edu for details.", "sources": []}
4. For time-sensitive content (deadlines, events), append: " (Please verify on the official website as this information may have updated.)"
5. For information shown as images (e.g. employer logos, salary charts), state: "The website displays this as an image, which is not in the text content. Please visit the source page directly."
6. NO speculation, NO \"in my opinion\", NO general data science knowledge — context only.

Output ONLY the JSON object. No prose before or after.
"""

USER_PROMPT_TEMPLATE = """Context excerpts:
{context}

Question: {question}

JSON answer:"""


In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
import json, re

llm = ChatAnthropic(model=CLAUDE_MODEL, max_tokens=1024, temperature=0)

def format_context(docs):
    """Format retrieved docs as numbered context blocks with URL footers."""
    blocks = []
    for i, d in enumerate(docs, 1):
        blocks.append(f"""[Excerpt {i}]
Page: {d.metadata.get('breadcrumb', d.metadata.get('breadcrumbs', ''))}
URL: {d.metadata.get('url', '')}
Content: {d.page_content}""")
    return "\n\n".join(blocks)

def parse_json_response(raw_text):
    """Robustly extract JSON dict from LLM response (handles fences/prose)."""
    # Try strict JSON parse
    txt = raw_text.strip()
    if txt.startswith("```"):
        # Strip code fence
        txt = re.sub(r"^```(?:json)?\n?", "", txt)
        txt = re.sub(r"\n?```$", "", txt)
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        # Try to find the first JSON object via regex
        m = re.search(r"\{[\s\S]*?\}\s*$", txt)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
    # Fallback: empty
    return {"answer": raw_text, "sources": []}

def run_rag(question, k=RETRIEVAL_K):
    """Single RAG turn — returns retrieval results + parsed answer."""
    retrieved = vector_db.similarity_search(question, k=k)
    context_str = format_context(retrieved)

    messages = [
        ("system", SYSTEM_PROMPT),
        ("user", USER_PROMPT_TEMPLATE.format(context=context_str, question=question)),
    ]
    resp = llm.invoke(messages)
    parsed = parse_json_response(resp.content)

    return {
        "question": question,
        "retrieved_chunks": retrieved,
        "retrieved_urls": [d.metadata.get("url", "") for d in retrieved],
        "raw_response": resp.content,
        "parsed_answer": parsed.get("answer", ""),
        "cited_sources": parsed.get("sources", []),
        "cited_urls": [s.get("url", "") for s in parsed.get("sources", [])],
    }

# Smoke test
demo = run_rag("What letters of recommendation are required?")
print("Q:", demo["question"])
print("A:", demo["parsed_answer"])
print("Cited URLs:", demo["cited_urls"])
print("\nTop-1 retrieved:", demo["retrieved_chunks"][0].metadata.get("breadcrumbs", "")[:80])


## 4. Run RAG over all 30 questions

In [ ]:
import pandas as pd
from tqdm import tqdm

records = []
for q in tqdm(eval_set, desc="RAG"):
    try:
        r = run_rag(q["question"])
        records.append({
            "qid": q["qid"],
            "category": q["category"],
            "sub_type": q["sub_type"],
            "difficulty": q["difficulty"],
            "question": q["question"],
            "ground_truth_answer": q["ground_truth_answer"],
            "gold_urls": q["gold_urls"],
            "expected_behavior": q["expected_behavior"],
            "rag_answer": r["parsed_answer"],
            "rag_cited_urls": r["cited_urls"],
            "retrieved_urls": r["retrieved_urls"],
            "retrieved_chunks_text": [d.page_content for d in r["retrieved_chunks"]],
        })
    except Exception as e:
        print(f"⚠ {q['qid']} failed: {e}")
        records.append({
            "qid": q["qid"], "category": q["category"], "sub_type": q["sub_type"],
            "question": q["question"], "ground_truth_answer": q["ground_truth_answer"],
            "gold_urls": q["gold_urls"], "expected_behavior": q["expected_behavior"],
            "rag_answer": f"ERROR: {e}", "rag_cited_urls": [], "retrieved_urls": [],
            "retrieved_chunks_text": [],
        })

df = pd.DataFrame(records)
print(f"✓ {len(df)} questions processed")
df.head(3)


## 5. Retrieval metrics — R@K based on `gold_url` ∈ retrieved chunk URLs

In [ ]:
def recall_at_k(retrieved_urls, gold_urls, k):
    if not gold_urls:
        return None   # out_of_scope questions have no gold; skip
    top_k = set(retrieved_urls[:k])
    return int(any(g in top_k for g in gold_urls))

for k in [1, 3, 5, 10]:
    df[f"R@{k}"] = df.apply(lambda r: recall_at_k(r["retrieved_urls"], r["gold_urls"], k), axis=1)

# Aggregate (skipping None for out_of_scope)
print("=== Retrieval Recall @ K (excluding out_of_scope) ===")
for k in [1, 3, 5, 10]:
    valid = df[f"R@{k}"].dropna()
    print(f"  R@{k}: {valid.mean():.3f}  (n={len(valid)})")

print("\n=== Per-category R@5 ===")
print(df[df["sub_type"] != "out_of_scope"].groupby("category")["R@5"].agg(["mean", "count"]).round(3))


## 6. URL Citation metrics — custom, based on `cited_urls` ∩ `gold_urls`

In [ ]:
def citation_metrics(cited, gold):
    if not gold:
        # out_of_scope: cited should be empty; recall=NaN, precision=1 if empty
        return None, (1.0 if not cited else 0.0), None
    cited_s, gold_s = set(cited), set(gold)
    if not cited_s:
        return 0.0, 0.0, 0.0
    inter = cited_s & gold_s
    recall = len(inter) / len(gold_s)
    precision = len(inter) / len(cited_s)
    f1 = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0.0
    return recall, precision, f1

df[["cit_recall", "cit_precision", "cit_f1"]] = df.apply(
    lambda r: citation_metrics(r["rag_cited_urls"], r["gold_urls"]),
    axis=1, result_type="expand"
)

print("=== URL Citation (excluding out_of_scope) ===")
mask = df["sub_type"] != "out_of_scope"
print(f"  citation_recall:    {df.loc[mask, 'cit_recall'].mean():.3f}")
print(f"  citation_precision: {df.loc[mask, 'cit_precision'].mean():.3f}")
print(f"  citation_f1:        {df.loc[mask, 'cit_f1'].mean():.3f}")

print("\n=== Per-category citation F1 ===")
print(df[mask].groupby("category")["cit_f1"].mean().round(3))


## 7. RAGAS evaluation (faithfulness, answer_relevancy, context_precision, context_recall)

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_anthropic import ChatAnthropic
from datasets import Dataset

# Filter out out_of_scope (RAGAS expects ground truth answer)
mask = df["sub_type"] != "out_of_scope"
sub = df[mask].copy()

ragas_data = {
    "question": sub["question"].tolist(),
    "answer": sub["rag_answer"].tolist(),
    "contexts": sub["retrieved_chunks_text"].tolist(),
    "ground_truth": sub["ground_truth_answer"].tolist(),
}

evaluator_llm = LangchainLLMWrapper(ChatAnthropic(
    model=CLAUDE_MODEL, timeout=120, max_retries=3, max_tokens=1024,
))
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

ragas_results = evaluate(
    Dataset.from_dict(ragas_data),
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False,
)
ragas_df = ragas_results.to_pandas()
print("=== RAGAS aggregate scores ===")
for m in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
    if m in ragas_df.columns:
        print(f"  {m}: {ragas_df[m].mean():.3f}")


In [ ]:
# Merge RAGAS scores back into df
ragas_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
for col in ragas_cols:
    if col in ragas_df.columns:
        sub[col] = ragas_df[col].values
df.loc[mask, ragas_cols] = sub[ragas_cols].values


## 8. Edge case handling — out_of_scope refusal + time/image caveat rate

In [ ]:
# Out-of-scope: did the RAG output the refusal template?
def is_refusal(answer):
    text = (answer or "").lower()
    refusal_signals = ["don't have", "do not have", "not in the program website",
                       "contact applieddatascience-admissions"]
    return any(s in text for s in refusal_signals)

oos = df[df["sub_type"] == "out_of_scope"]
if len(oos):
    refusal_rate = oos["rag_answer"].apply(is_refusal).mean()
    print(f"Out-of-scope refusal rate: {refusal_rate:.0%} ({len(oos)} questions)")

# Time-sensitive: did it include the caveat?
def has_time_caveat(answer):
    text = (answer or "").lower()
    return any(s in text for s in ["verify on", "may have updated", "may have changed", "official website"])

ts = df[df["sub_type"] == "time_sensitive"]
if len(ts):
    caveat_rate = ts["rag_answer"].apply(has_time_caveat).mean()
    print(f"Time-sensitive caveat rate: {caveat_rate:.0%} ({len(ts)} questions)")

# Image-only: did it mention images?
def has_image_caveat(answer):
    text = (answer or "").lower()
    return any(s in text for s in ["image", "displays this", "logos", "chart"])

img = df[df["sub_type"] == "image_only"]
if len(img):
    img_rate = img["rag_answer"].apply(has_image_caveat).mean()
    print(f"Image-only caveat rate: {img_rate:.0%} ({len(img)} questions)")


## 9. Failure analysis — which questions failed & why

In [ ]:
# Find failures: low R@5 (retrieval miss) or low citation_f1 (cite wrong)
failures = df[
    ((df["R@5"] == 0) | (df["cit_f1"] < 0.5)) & (df["sub_type"] != "out_of_scope")
].copy()

if len(failures):
    print(f"⚠ {len(failures)} failure cases")
    for _, r in failures.iterrows():
        print(f"\n[{r['qid']} / {r['category']} / {r['sub_type']} / diff={r['difficulty']}]")
        print(f"  Q: {r['question']}")
        print(f"  Gold URLs: {r['gold_urls']}")
        print(f"  Top retrieved URLs: {r['retrieved_urls'][:3]}")
        print(f"  R@5={r['R@5']} cit_f1={r['cit_f1']:.2f}")
else:
    print("✓ No major failures")


## 10. Save outputs

In [ ]:
# Per-question CSV
out_cols = [
    "qid", "category", "sub_type", "difficulty", "question",
    "ground_truth_answer", "rag_answer",
    "gold_urls", "rag_cited_urls",
    "R@1", "R@3", "R@5", "R@10",
    "cit_recall", "cit_precision", "cit_f1",
    "faithfulness", "answer_relevancy", "context_precision", "context_recall",
]
out_cols = [c for c in out_cols if c in df.columns]
df[out_cols].to_csv(RESULTS_CSV, index=False)
print(f"✓ Wrote {RESULTS_CSV}")


In [ ]:
# Summary markdown
lines = ["# MSADS RAG Evaluation Summary\n"]

lines.append("## Aggregate metrics\n")
mask = df["sub_type"] != "out_of_scope"
agg = {
    "Retrieval R@1": df.loc[mask, "R@1"].mean(),
    "Retrieval R@3": df.loc[mask, "R@3"].mean(),
    "Retrieval R@5": df.loc[mask, "R@5"].mean(),
    "Retrieval R@10": df.loc[mask, "R@10"].mean(),
    "Citation Recall": df.loc[mask, "cit_recall"].mean(),
    "Citation Precision": df.loc[mask, "cit_precision"].mean(),
    "Citation F1": df.loc[mask, "cit_f1"].mean(),
}
for m in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
    if m in df.columns:
        agg[f"RAGAS {m}"] = df.loc[mask, m].mean()
lines.append("| Metric | Score |")
lines.append("|---|---|")
for k, v in agg.items():
    lines.append(f"| {k} | {v:.3f} |")
lines.append("")

lines.append("## Per-category breakdown\n")
cat_cols = ["R@5", "cit_f1"]
cat_cols += [c for c in ["faithfulness", "answer_relevancy"] if c in df.columns]
breakdown = df[mask].groupby("category")[cat_cols].mean().round(3)
lines.append(breakdown.to_markdown())
lines.append("")

with open(SUMMARY_MD, "w") as f:
    f.write("\n".join(lines))
print(f"✓ Wrote {SUMMARY_MD}")


## Done

Deliverables:
- `eval_results.csv` — per-question results (30 rows × ~15 columns)
- `eval_summary.md` — aggregate + per-category breakdown for the writeup

Next: include both in the 5-page midterm doc + reference in the 10-min PPT.